# Notebook 3: Generate Per-Organism HTML Pages

Reads the cached SRA and BacDive CSVs produced by notebooks 1 and 2, then generates
one standalone HTML page per organism and a searchable index page.

Each organism page contains:
- PlasticDB summary: plastics degraded, thermophilic flag, evidence badges, isolation info
- Full PlasticDB entries table with plastic, year, enzyme, sequence/GenBank flags, DOI
- NCBI SRA section: run count, platforms, library strategies, total bases, year range
- BacDive section: culture temperature, pH, oxygen tolerance, Gram stain, morphology, isolation source

All data shown is real and sourced from the cached CSVs. Fields with no data are labeled 'Not recorded' or 'Not found'. Nothing is estimated or interpolated.

Output: `../pages/index.html` and `../pages/<slug>.html` for each organism.

In [ ]:
import re
import unicodedata
from pathlib import Path

import pandas as pd

ROOT        = Path.cwd().parent
PDB_TSV     = ROOT.parent / 'plastic-biodegradation-analysis' / 'data' / 'plasticdb_microorganisms.tsv'
SRA_CSV     = ROOT / 'data' / 'sra_stats.csv'
BACDIVE_CSV = ROOT / 'data' / 'bacdive_data.csv'
PAGES_DIR   = ROOT / 'pages'
PAGES_DIR.mkdir(exist_ok=True)

print('Paths ready.')

In [ ]:
# Load all data
raw = pd.read_csv(PDB_TSV, sep='\t', dtype=str, on_bad_lines='skip')
raw.columns = [
    'organism','tax_id','plastic','reference','enzyme_name',
    'enzyme_id','db_enzyme_name','gene','genbank_id','sequence',
    'year','evidence','plastic_used','manufacturer','analytical_grade',
    'thermophilic','isolation_sample','isolation_environment',
    'isolation_location','extrapolated_from_enzyme','enzyme_id_in_paper','doi',
]
raw['organism']     = raw['organism'].str.strip()
raw['has_sequence'] = raw['sequence'].notna() & (raw['sequence'].str.len() > 10)
raw['has_genbank']  = raw['genbank_id'].notna() & (raw['genbank_id'].str.len() > 3)
raw['year']         = pd.to_numeric(raw['year'], errors='coerce')

sra_df = pd.read_csv(SRA_CSV, dtype=str) if SRA_CSV.exists() else pd.DataFrame()
bd_df  = pd.read_csv(BACDIVE_CSV, dtype=str) if BACDIVE_CSV.exists() else pd.DataFrame()

print(f'PlasticDB entries:   {len(raw)}')
print(f'SRA rows cached:     {len(sra_df)}')
print(f'BacDive rows cached: {len(bd_df)}')

In [ ]:
# Build organism summary table
orgs = (
    raw.groupby('organism')
    .agg(
        tax_id         = ('tax_id', 'first'),
        n_entries      = ('organism', 'count'),
        plastics       = ('plastic', lambda x: sorted(x.dropna().unique().tolist())),
        n_plastics     = ('plastic', 'nunique'),
        has_sequence   = ('has_sequence', 'any'),
        has_genbank    = ('has_genbank', 'any'),
        has_enzyme     = ('enzyme_name', lambda x: x.notna().any()),
        first_year     = ('year', 'min'),
        last_year      = ('year', 'max'),
        thermophilic   = ('thermophilic', lambda x: x.mode()[0] if len(x.mode()) else ''),
        isolation_envs = ('isolation_environment', lambda x: '; '.join(sorted(x.dropna().unique()))),
        isolation_locs = ('isolation_location',    lambda x: '; '.join(sorted(x.dropna().unique()))),
    )
    .reset_index()
)

sra_idx = sra_df.set_index('organism') if not sra_df.empty else pd.DataFrame()
bd_idx  = bd_df.set_index('organism')  if not bd_df.empty  else pd.DataFrame()

print(f'Organisms: {len(orgs)}')

In [ ]:
# Helper functions
def slug(name):
    name = unicodedata.normalize('NFKD', name).encode('ascii','ignore').decode()
    return re.sub(r'[^a-z0-9]+', '_', name.lower()).strip('_')

def badge(val):
    s = str(val).lower()
    if s in ('true','yes','1'): return '<span class="badge badge-yes">Yes</span>'
    if s in ('false','no','0'): return '<span class="badge badge-no">No</span>'
    return '<span class="badge badge-na">N/A</span>'

def fmt_bases(b):
    try:
        b = float(b)
        if b >= 1e12: return f'{b/1e12:.2f} Tbp'
        if b >= 1e9:  return f'{b/1e9:.2f} Gbp'
        if b >= 1e6:  return f'{b/1e6:.2f} Mbp'
        return f'{b:.0f} bp'
    except: return 'N/A'

def kv(label, value, pill=False):
    if pill and value:
        pills = ''.join(f'<span class="pill">{v.strip()}</span>'
                        for v in str(value).split(';') if v.strip())
        return f'<div class="kv-row"><span class="kv-label">{label}</span><span class="kv-value">{pills}</span></div>'
    return f'<div class="kv-row"><span class="kv-label">{label}</span><span class="kv-value">{value or "N/A"}</span></div>'

print('Helpers defined.')

In [ ]:
CSS = '''
* { box-sizing: border-box; margin: 0; padding: 0; }
body { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
       background: #f8f9fa; color: #1a1a2e; line-height: 1.6; }
.header { background: #1a1a2e; color: #fff; padding: 24px 40px; }
.header h1 { font-size: 1.9rem; font-weight: 700; }
.header .meta { color: #9fb3cc; font-size: 0.9rem; margin-top: 6px; }
.breadcrumb { padding: 12px 40px; background: #eef1f5; font-size: 0.85rem; }
.breadcrumb a { color: #3a7bd5; text-decoration: none; }
.container { max-width: 1100px; margin: 0 auto; padding: 32px 40px; }
.section { background: #fff; border-radius: 8px; border: 1px solid #dde3ec;
           margin-bottom: 24px; overflow: hidden; }
.section-header { background: #f0f4fb; border-bottom: 1px solid #dde3ec;
                  padding: 14px 22px; font-weight: 600; font-size: 1rem; }
.section-body { padding: 20px 22px; }
.grid2 { display: grid; grid-template-columns: 1fr 1fr; gap: 18px; }
.kv-row { display: flex; gap: 12px; margin-bottom: 10px; font-size: 0.92rem; }
.kv-label { color: #6b7a99; min-width: 180px; flex-shrink: 0; }
.kv-value { font-weight: 500; }
.pill { display: inline-block; background: #e8f0fe; color: #3a5cc0;
        border-radius: 4px; padding: 2px 8px; font-size: 0.8rem; margin: 2px; }
.badge { display: inline-block; border-radius: 4px; padding: 2px 8px;
         font-size: 0.78rem; font-weight: 600; margin-right: 4px; }
.badge-yes { background: #d4edda; color: #155724; }
.badge-no  { background: #f8d7da; color: #721c24; }
.badge-na  { background: #e2e3e5; color: #383d41; }
table { width: 100%; border-collapse: collapse; font-size: 0.88rem; }
th { background: #f0f4fb; padding: 10px 14px; text-align: left;
     font-weight: 600; border-bottom: 2px solid #dde3ec; }
td { padding: 9px 14px; border-bottom: 1px solid #edf0f5; vertical-align: top; }
.stat-box { background: #f0f4fb; border-radius: 6px; padding: 16px; text-align: center; }
.stat-val { font-size: 1.8rem; font-weight: 700; color: #3a5cc0; }
.stat-lbl { font-size: 0.8rem; color: #6b7a99; margin-top: 4px; }
.no-data  { color: #9aa3b5; font-style: italic; font-size: 0.9rem; }
a { color: #3a7bd5; }
#search-input { width:100%;padding:10px 14px;font-size:1rem;
  border:1px solid #ccd3e0;border-radius:6px;margin-bottom:16px;outline:none; }
'''

def build_page(org_row, full_df, sra_row, bd_row):
    name = org_row['organism']
    ent  = full_df[full_df['organism'] == name].copy()

    # Entries table
    rows_html = ''
    for _, e in ent.iterrows():
        doi = str(e.get('doi','') or '')
        doi_link = (f'<a href="https://doi.org/{doi}" target="_blank">{doi[:40]}</a>'
                    if len(doi) > 5 else doi)
        yr = int(e['year']) if pd.notna(e.get('year')) else 'N/A'
        rows_html += (f'<tr><td>{e.get("plastic","")}</td><td>{yr}</td>'
                      f'<td>{str(e.get("enzyme_name","") or "")[:50] or "N/A"}</td>'
                      f'<td>{badge(e.get("has_sequence",False))}</td>'
                      f'<td>{badge(e.get("has_genbank",False))}</td>'
                      f'<td style="font-size:0.8rem">{doi_link}</td></tr>')

    entries_html = (f'<table><thead><tr><th>Plastic</th><th>Year</th><th>Enzyme</th>'
                    f'<th>Sequence</th><th>GenBank</th><th>DOI</th></tr></thead>'
                    f'<tbody>{rows_html}</tbody></table>')

    # SRA block
    if sra_row is not None:
        try: rc = int(float(sra_row.get('sra_run_count','')))
        except: rc = 'N/A'
        sra_html = (f'<div class="grid2" style="margin-bottom:18px">'
                    f'<div class="stat-box"><div class="stat-val">{rc}</div>'
                    f'<div class="stat-lbl">SRA runs deposited</div></div>'
                    f'<div class="stat-box"><div class="stat-val">{fmt_bases(sra_row.get("sra_total_bases"))}</div>'
                    f'<div class="stat-lbl">Total bases (sampled)</div></div></div>'
                    + kv('Platforms',       sra_row.get('sra_platforms','')  or 'None in sample', pill=True)
                    + kv('Strategies',      sra_row.get('sra_strategies','') or 'None in sample', pill=True)
                    + kv('Deposit years',   sra_row.get('sra_date_range','') or 'N/A')
                    + '<p style="font-size:0.8rem;color:#9aa3b5;margin-top:10px">'
                    'Source: NCBI SRA E-utilities esearch + esummary. '
                    'Platform/strategy sampled from up to 200 runs.</p>')
    else:
        sra_html = '<p class="no-data">SRA data not fetched for this organism.</p>'

    # BacDive block
    if bd_row is not None and str(bd_row.get('bacdive_found','')).lower() == 'yes':
        bd_url = bd_row.get('bacdive_url','')
        link   = (f'<a href="{bd_url}" target="_blank">BacDive strain {bd_row.get("bacdive_strain_id","")}</a>'
                  if bd_url else 'N/A')
        bd_html = (kv('BacDive record',       link)
                   + kv('Culture temp (C)',   bd_row.get('bacdive_temp_c','')   or 'Not recorded')
                   + kv('pH',                bd_row.get('bacdive_ph','')        or 'Not recorded')
                   + kv('Oxygen tolerance',  bd_row.get('bacdive_oxygen','')    or 'Not recorded')
                   + kv('Gram stain',        bd_row.get('bacdive_gram','')      or 'Not recorded')
                   + kv('Morphology',        bd_row.get('bacdive_morphology','') or 'Not recorded')
                   + kv('Motility',          bd_row.get('bacdive_motility','')  or 'Not recorded')
                   + kv('Isolation source',  bd_row.get('bacdive_isolation','') or 'Not recorded')
                   + '<p style="font-size:0.8rem;color:#9aa3b5;margin-top:10px">'
                   'Source: BacDive public strain page (bacdive.dsmz.de).</p>')
    elif bd_row is not None:
        bd_html = '<p class="no-data">Organism not found in BacDive public database.</p>'
    else:
        bd_html = '<p class="no-data">BacDive data not fetched for this organism.</p>'

    plastics = org_row.get('plastics',[])
    if isinstance(plastics, str):
        import ast
        try: plastics = ast.literal_eval(plastics)
        except: plastics = [p.strip() for p in plastics.split(',')]
    pills = ''.join(f'<span class="pill">{p}</span>' for p in plastics)

    fy = int(org_row['first_year']) if pd.notna(org_row.get('first_year')) else 'N/A'
    ly = int(org_row['last_year'])  if pd.notna(org_row.get('last_year'))  else 'N/A'

    return f'''<!DOCTYPE html><html lang="en">
<head><meta charset="UTF-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>{name} | Organism Profile</title><style>{CSS}</style></head><body>
<div class="header"><h1>{name}</h1><div class="meta">
Tax ID: {org_row.get('tax_id','N/A')} &nbsp;|&nbsp;
{org_row.get('n_plastics',0)} plastic type(s) &nbsp;|&nbsp;
{org_row.get('n_entries',0)} PlasticDB entries &nbsp;|&nbsp;
Years: {fy} to {ly}
</div></div>
<div class="breadcrumb"><a href="index.html">All organisms</a> / {name}</div>
<div class="container">
<div class="section"><div class="section-header">PlasticDB Summary</div>
<div class="section-body"><div class="grid2"><div>
{kv('Plastics degraded', pills)}
{kv('Thermophilic flag', org_row.get('thermophilic','') or 'Not recorded')}
{kv('Has linked sequence', badge(org_row.get('has_sequence')))}
{kv('Has named enzyme',    badge(org_row.get('has_enzyme')))}
{kv('Has GenBank ID',      badge(org_row.get('has_genbank')))}
</div><div>
{kv('Isolation environments', org_row.get('isolation_envs','') or 'Not recorded', pill=True)}
{kv('Isolation locations',    org_row.get('isolation_locs','') or 'Not recorded', pill=True)}
</div></div></div></div>
<div class="section"><div class="section-header">PlasticDB Entries ({org_row.get('n_entries',0)} total)</div>
<div class="section-body">{entries_html}</div></div>
<div class="section"><div class="section-header">NCBI SRA Sequencing Data</div>
<div class="section-body">{sra_html}</div></div>
<div class="section"><div class="section-header">BacDive Physiological Data</div>
<div class="section-body">{bd_html}</div></div>
</div></body></html>'''

print('build_page() defined.')

In [ ]:
# Generate all organism pages
for i, (_, row) in enumerate(orgs.iterrows(), 1):
    name    = row['organism']
    s       = slug(name)
    sra_row = sra_idx.loc[name] if (not sra_idx.empty and name in sra_idx.index) else None
    bd_row  = bd_idx.loc[name]  if (not bd_idx.empty  and name in bd_idx.index)  else None
    html    = build_page(row, raw, sra_row, bd_row)
    (PAGES_DIR / f'{s}.html').write_text(html, encoding='utf-8')
    if i % 100 == 0:
        print(f'  {i}/{len(orgs)} pages written')

print(f'All {len(orgs)} pages written.')

In [ ]:
# Generate index.html
sra_map = dict(zip(sra_df['organism'], sra_df.get('sra_run_count', pd.Series(dtype=str)))) if not sra_df.empty else {}
bd_map  = dict(zip(bd_df['organism'],  bd_df.get('bacdive_found',  pd.Series(dtype=str)))) if not bd_df.empty  else {}

rows_html = ''
for _, row in orgs.sort_values('n_entries', ascending=False).iterrows():
    org = row['organism']
    s   = slug(org)
    try: rc = int(float(sra_map.get(org,'')))
    except: rc = 'N/A'
    plastics = row.get('plastics',[])
    if isinstance(plastics, str):
        import ast
        try: plastics = ast.literal_eval(plastics)
        except: plastics = [p.strip() for p in plastics.split(',')]
    pills = ''.join(f'<span class="pill">{p}</span>' for p in plastics[:5])
    if len(plastics) > 5: pills += f'<span class="pill">+{len(plastics)-5}</span>'
    bd_found = str(bd_map.get(org,'')).lower() == 'yes'
    fy = int(row['first_year']) if pd.notna(row.get('first_year')) else 'N/A'
    rows_html += (f'<tr><td><a href="{s}.html">{org}</a></td>'
                  f'<td style="text-align:center">{row.get("n_entries",0)}</td>'
                  f'<td>{pills}</td>'
                  f'<td style="text-align:center">{rc}</td>'
                  f'<td style="text-align:center">{\'<span class="badge badge-yes">Yes</span>\' if bd_found else \'<span class="badge badge-no">No</span>\'}</td>'
                  f'<td style="text-align:center">{fy}</td></tr>')

index_html = f'''<!DOCTYPE html><html lang="en">
<head><meta charset="UTF-8"><title>Organism Profiles</title><style>{CSS}</style></head><body>
<div class="header"><h1>Organism Profiles</h1>
<div class="meta">{len(orgs)} organisms. PlasticDB + NCBI SRA + BacDive. Real data only.</div></div>
<div class="container"><div class="section"><div class="section-header">All Organisms</div>
<div class="section-body">
<input id="search-input" placeholder="Filter by organism name..." oninput="filterTable(this.value)">
<table id="org-table"><thead><tr>
<th>Organism</th><th style="text-align:center">PlasticDB entries</th>
<th>Plastics</th><th style="text-align:center">SRA runs</th>
<th style="text-align:center">BacDive</th><th style="text-align:center">First year</th>
</tr></thead><tbody>{rows_html}</tbody></table>
</div></div></div>
<script>function filterTable(q){{
  q=q.toLowerCase();
  document.querySelectorAll('#org-table tbody tr').forEach(r=>{{
    r.style.display=r.cells[0].textContent.toLowerCase().includes(q)?'':'none';
  }});
}}</script></body></html>'''

(PAGES_DIR / 'index.html').write_text(index_html, encoding='utf-8')
print(f'index.html written. Total pages: {len(list(PAGES_DIR.glob("*.html")))}')